# 지금까지의 이미지 EDA

이 노트북은 **기존 카탈로그와 판독 기록을 읽는 로컬 탐색 화면**입니다. 처음부터 전수 `profile`을 다시 실행하지 않습니다.

- 원본 이미지/CSV와 판독 원응답을 수정하지 않습니다. 모델 학습·원격 전송·가중치 다운로드를 하지 않습니다.
- 공개용 원본 노트북은 출력이 비어 있습니다. 데이터가 포함된 실행본·HTML은 `runs/`에만 저장하세요.
- 표와 그래프의 설명을 읽은 다음 셀을 실행하세요. **무엇을 보았는지**와 **불량 정답을 확인했는지**는 다릅니다.
- 직전에 만든 와셔 포스터는 교육용 가상 그림입니다. 이 노트북의 실제 이미지와 별개입니다.

로컬 실행: `python -m pip install -e '.[notebook]'` 후 `jupyter lab notebooks/eda.ipynb`.
Colab에서는 저장소와 비공개 `runs/` 자료를 본인 환경에 준비하고 아래 경로를 바꾸세요. 무료 GPU는 필요하지 않습니다.

## 0. 전체 지도: 우리는 무엇을 배우려 하나요?

**최종 과제:** 여러 제품 사진에서 이미지당 정상/불량 점수 하나를 만든다.

```text
과제 정의: 사진 한 장에 어떤 정답이 붙는가?
  ├─ 데이터 구성: 제품 종류,0/1 라벨,제품 개수
  ├─ 관측 가능성: 파일 읽기,화질,반사,가림,작은 정보
  ├─ 관계와 평가: 같은 촬영/개체,중복,학습·검증 분리
  └─ 학습과 측정: 모델·학습 가중치,AUROC,재현율·정밀도
```

**EDA(탐색적 데이터 분석)**는 위 질문을 확인해 입력 변환·분할·작은 실험을 정당화하는 과정입니다. 눈으로 사진을 보는 것만도 아니고, 수치 그래프를 많이 만드는 것만도 아닙니다.

읽는 순서: 전체 구성(3–4절) → 읽기와 품질의 한계(2·5절) → 실제 관측과 용어(6–9절) → 다음 결정(10절).
먼저 코드보다 각 Markdown 설명을 읽고, 그다음 표/그래프를 확인하세요.

## 1. 읽을 파일과 이미지 표시 선택

기본 경로는 저장소 안의 비공개 `runs/`입니다. 파일이 없으면 생성하거나 다운로드하지 않고 안내 후 멈춥니다.
사진은 **이미 판독한 학습 표본의 익명 원본 사본**만 표시합니다. 테스트 사진이나 기존 `contact_sheet.jpg`는 열지 않습니다.
`SHOW_IMAGES`를 켜면 아래의 최대 6개 사진이 실행 출력에 포함됩니다. 원래 라벨·파일명 대응표는 읽지 않습니다.

### 이 절은 ‘실험 준비’에 속합니다
- **카탈로그(profile)**: 사진을 한 장씩 분석한 크기·색상·파일 해시 등의 목록입니다. 사진 자체나 모델 가중치가 아닙니다.
- **판독 기록(observations)**: 사람이든 AI든 화면을 보고 당시 무엇이 보였다고 적었는지를 보존한 기록입니다.
- **환경변수/경로**: 어느 비공개 파일을 읽을지 정하는 설정입니다. 여기서 모델의 판단 기준을 학습하지 않습니다.
- `SHOW_IMAGES`: 이미 본 사진을 화면에 다시 표시하는 스위치입니다. 켠다고 새 표본이나 라벨이 생기지 않습니다.

In [ ]:
import os
import sys
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import matplotlib.pyplot as plt
from IPython import get_ipython
from IPython.display import display, Markdown, Image as DisplayImage

candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for p in candidates if (p / "src/defectlab").is_dir()), None)
if os.environ.get("DEFECTLAB_REPO_ROOT"):
    REPO_ROOT = Path(os.environ["DEFECTLAB_REPO_ROOT"]).expanduser().resolve()
if REPO_ROOT is None:
    raise FileNotFoundError("저장소에서 열거나 DEFECTLAB_REPO_ROOT를 지정하세요.")
PROFILE_PATH = REPO_ROOT / os.environ.get("DEFECTLAB_EDA_PROFILE", "runs/real-eda/profile.json")
OBS_DIR = REPO_ROOT / os.environ.get(
    "DEFECTLAB_EDA_OBSERVATIONS", "runs/pro-assisted-eda/visual-calibration-001"
)
SHOW_IMAGES = os.environ.get("DEFECTLAB_EDA_SHOW_IMAGES", "0") == "1"
PHOTO_IDS = []  # 비우면 기존 첫 판독의 앞 6개 익명 ID. 추가 표본은 뽑지 않습니다.
MAX_PHOTOS = 6

sys.path.insert(0, str(REPO_ROOT / "src"))
from defectlab.common import hash_json  # noqa: E402 — 위에서 로컬 저장소 경로를 등록한 뒤 가져옵니다.

shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
# 그래프 안은 글꼴 의존성을 줄이기 위해 영어, 해설과 표는 한국어를 씁니다.
INPUT_HASHES = {}
def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def read_evidence(path, required=False):
    if not path.is_file():
        if required:
            raise FileNotFoundError(f"기존 근거 파일이 없습니다: {path.name}. 자동 재분석하지 않습니다.")
        return None
    INPUT_HASHES[str(path.relative_to(REPO_ROOT)) if path.is_relative_to(REPO_ROOT) else path.name] = sha256_file(path)
    return json.loads(path.read_text(encoding="utf-8"))

print("이미지 표시:", SHOW_IMAGES, "| GPU/원격 연결 사용: 없음")

## 2. 카탈로그 무결성과 확인 범위

이 셀은 **카탈로그 내부 해시**를 확인합니다. 원본 11,621장을 현재 다시 디코드하거나 원본 바이트를 검증하는 셀이 아닙니다.
데이터가 바뀌었는지는 별도 원본 manifest 검사로 확인해야 합니다. 실행 날짜도 과거 조사 날짜와 구분합니다.

### 이 절은 ‘근거가 같은 파일인가?’를 확인합니다
**해시(hash)**는 파일 내용의 지문입니다. 내용이 바뀌면 지문도 바뀝니다. 카탈로그 내부 해시 통과는 “저장된 해시와 현재 내용이 일치하는가”를 확인하는 것이며 “기록의 해석이 옳다”를 인증하지 않습니다.
`현재 원본 바이트 미재검사`는 지금 이 노트북이 모든 원본을 다시 읽지는 않았다는 뜻입니다. 과거 검사 결과를 버렸다는 뜻도 아닙니다.

In [ ]:
profile = read_evidence(PROFILE_PATH, required=True)
expected = profile.get("sha256")
actual = hash_json({k: v for k, v in profile.items() if k != "sha256"})
if not expected or actual != expected:
    raise ValueError("카탈로그 내부 해시 불일치: 기존 근거를 수정하거나 그대로 신뢰하지 마세요.")
df = pd.DataFrame(profile["records"])
required_columns = {"image_path", "relative_path", "role", "label", "width", "height", "sha256"}
if not required_columns.issubset(df.columns):
    raise ValueError("카탈로그 필수 열이 없습니다.")
if df["image_path"].duplicated().any() or not df["role"].isin(["train", "test"]).all():
    raise ValueError("이미지 ID 중복 또는 알 수 없는 role이 있습니다.")
train = df.loc[df.role.eq("train")].copy()
test = df.loc[df.role.eq("test")].copy()
if not train.label.isin([0, 1]).all() or not test.label.isna().all():
    raise ValueError("학습 라벨 또는 테스트 무라벨 계약과 다릅니다.")
for part in [train, test]:
    part["size_raw"] = part.width.astype(str) + " × " + part.height.astype(str)
    part["aspect_raw"] = part.width / part.height
    part["mean_rgb"] = part.rgb_mean.map(lambda v: sum(v) / len(v) if isinstance(v, list) and v else float("nan"))

display(pd.DataFrame([
    {"확인 항목": "카탈로그 파일/내부 해시", "상태": "현재 파일 확인 통과"},
    {"확인 항목": "원본 바이트 현재 일치", "상태": "이 노트북에서는 재검사하지 않음"},
    {"확인 항목": "전체 디코드", "상태": "과거 카탈로그의 검사 결과를 읽음"},
    {"확인 항목": "실제 촬영/제품 그룹 독립성", "상태": "미확인"},
]))
print("카탈로그 행:", len(df), "| dataset hash:", profile["dataset_hash"])
print("이번 실행:", datetime.now(timezone.utc).isoformat())

## 3. 전체 수와 클래스 불균형

`0=정상`, `1=불량`입니다. 막대는 **학습 이미지 수**이며 제품 개수나 결함 영역 수가 아닙니다.
다수 클래스가 많다는 사실만으로 라벨 오류나 성능을 판단할 수 없습니다.

### 이 절은 ‘라벨 분포와 평가’를 다룹니다
라벨0/1은 **우리가 맞혀야 할 답(target)**입니다. 제품 종류나 이미지 파일의 정상 작동 여부를 뜻하지 않습니다. 여러 제품 종류가 있어도 출력 목표는 이미지당 정상/불량 점수 하나일 수 있습니다.

현재 정상:불량은 약8:1입니다. 중요하지만 이 숫자만으로 “반드시 증강/감소해야 한다”는 결론은 나오지 않습니다. 같은 불량이라도 특정 제품·촬영·결함 형태에 몰렸는지가 더 중요할 수 있습니다.

**정확도(accuracy)**는 전체 중 맞힌 비율입니다. 무조건 정상이라고 하면 불량을 하나도 못 찾아도 높은 정확도가 나옵니다.
**불량 재현율(recall)**은 실제 불량 중 찾아낸 비율입니다. 검증 정답과 이진 판단이 있으면 계산할 수 있습니다. 실제 불량100장 중70장을 찾으면70%입니다. 지금은 실제 모델 예측이 없어 측정하지 않은 것입니다.
**정밀도(precision)**는 불량이라고 경고한 것 중 실제 불량 비율입니다. 모두 불량이라고 하면 재현율100%여도 정상 오경고가 많을 수 있습니다.
**AUROC**는 점수의 순위 구분력을 봅니다. 모든 이미지에 같은 점수를 주면 두 클래스가 있는 경우0.5입니다. 대회 주 지표는 전체 검증 이미지 AUROC이고, 정확도·재현율과 같은 숫자가 아닙니다.

점수→0/1로 바꾸는 **임계값(threshold)**을 낮추면 일반적으로 더 많이 불량으로 경고합니다. 재현율만 올리고 정상 오경고도 늘릴 수 있으므로 정밀도·정상 오경고를 함께 봅니다. sigmoid 점수0.5가 최적 기준이나 검증된 확률이라는 보장은 없습니다.

공식 설명: [재현율](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html), [ROC-AUC](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html).

In [ ]:
counts = train.label.astype(int).value_counts().reindex([0, 1], fill_value=0)
summary = pd.DataFrame({"라벨": ["0 정상", "1 불량"], "이미지 수": counts.to_numpy()})
summary["학습 내 비율"] = summary["이미지 수"] / len(train)
display(summary)
print("학습:", len(train), "| 테스트:", len(test), "(정답 없음)")
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(["Normal (0)", "Defective (1)"], counts.to_numpy(), color=["#268b8d", "#d79635"])
ax.set(ylabel="Training images", title="Image-label counts")
plt.show()

### 3-A. 불균형이면 바로 증강하거나 정상 사진을 줄이나요?

그렇게 확정하지 않습니다. 먼저 같은 검증 분할에서 기본 학습의 점수·오류를 확인합니다. 이후 한 가지씩 비교합니다.

| 후보 | 무엇을 바꾸는가 | 비용·한계 |
|---|---|---|
| 손실 가중치 | 불량을 틀렸을 때 학습 손실에 더 큰 비중 | 실제 새 결함 정보가 생기지는 않음 |
| 균형 샘플링 | 학습 배치에 불량이 더 자주 나오도록 선택 | 같은 사례 반복으로 과적합할 수 있음 |
| 일반/생성 증강 | 원래 학습 사진의 허용된 변형 추가 | 결함이 사라지거나 생성 흔적을 배울 수 있음 |
| 정상 감소 | 정상 학습 사진 일부만 사용 | 정상 다양성을 잃을 수 있어 처음부터 전량을 버리지 않음 |

현재8:1 비율만으로 우승 방법을 고를 수 없습니다. 가중치·샘플링·증강을 한꺼번에 바꾸면 무엇이 효과를 냈는지도 알기 어렵습니다.
**분할을 먼저 정하고 학습 부분에만 적용합니다. 검증셋을 억지로1:1로 맞추거나 그 원본을 증강 참조에 쓰지 않습니다.**
참고: [PyTorch positive weight](https://docs.pytorch.org/docs/2.15/generated/torch.nn.BCEWithLogitsLoss.html), [리샘플링과 누수](https://imbalanced-learn.org/stable/common_pitfalls.html).

### 3-B. 제품이 여러 종류인데 라벨은0/1뿐이라면?

| 정보 | 예 | 현재 정답으로 확인됐나요? |
|---|---|---|
| 학습 목표 라벨 | 정상0/불량1 | 제공 CSV의 목표 라벨 |
| 제품 유형 | 기판류/식품처럼 보이는 외형 | 공식 범주·정확한 이름은 미확인 |
| 물리적 개체 | 같은 실물A의 여러 촬영 | 미확인 |
| 정상·불량 쌍 | 같은 실물을 결함 전후로 촬영 | 미확인 |
| 사진의 제품 수 | one/multiple | 조정 전 시각 관측 |

여러 유형을 한 모델이 보고 공통으로 정상/불량을 배우는 것은 가능합니다. 0/1만으로 제품명을 학습시키는 정답이나 결함 위치 정답이 자동으로 생기지는 않습니다.
같은 종류의 정상·불량 사진이 있다고 **같은 실물의 전후 쌍**인 것은 아닙니다. paired 학습을 전제하지 않습니다.

다음 조사에서는 관측용 제품군 후보를 검토하고, 블라인드 관측을 고정한 후 별도 규약으로 제품군×0/1 분포를 대조할 수 있습니다. 어떤 제품군이 거의 한 라벨에만 있으면 모델이 결함보다 제품 종류를 외우는지 의심할 질문이 생깁니다. 아직 그 대조를 수행하거나 확인한 결과는 아닙니다.
제품군별 모델·정상 reference 비교·지역 점수 집계도 후보지만, 제품군을 확정하고 기본 모델을 측정하기 전에 결정하지 않습니다.

In [ ]:
# 가상 규칙의 지표 예시입니다. 실제 모델 추론/학습 결과가 아닙니다.
normal_n = int(counts.loc[0])
defective_n = int(counts.loc[1])
display(pd.DataFrame([
    {"가상 규칙": "무조건 정상", "정확도": normal_n / len(train),
     "불량 재현율": 0.0 if defective_n else None,
     "AUROC": 0.5 if normal_n and defective_n else None},
]))
print("0/1을 동일 확률로 무작위 선택할 때 기대 정확도는 50%입니다. 항상 정상이라는 규칙은 다른 비교입니다.")
print("이 표는 데이터 구성으로 계산한 예시이며 대회 모델 성능이 아닙니다.")

## 4. 해상도·비율과 전처리 질문

층화 표본은 원래 `width/height` 기준입니다. EXIF 보정 후 크기는 별도 열이며 조용히 섞지 않습니다.
크기와 라벨의 연관은 촬영·제품 구성의 단서일 수 있지만 **인과관계 또는 모델 의존성의 증명은 아닙니다.**
테스트는 입력 메타데이터만 비교하며 사진이나 추정 정답은 다루지 않습니다.

### 이 절은 ‘사진 모양과 입력 변환’을 다룹니다
**해상도**는 픽셀 격자의 가로×세로 크기입니다. 예를 들어1200×800은 가로1200개, 세로800개의 픽셀을 뜻합니다.
**aspect ratio(종횡비)**는 `가로 ÷ 세로`입니다. 1200×800은1.5, 800×1200은약0.67, 1000×1000은1입니다. 같은 비율이라도 픽셀 수는 다를 수 있습니다. 비율이1에 가깝다고 품질이 좋다는 뜻도 아닙니다.

서로 다른 비율을 정사각형으로 억지로 늘리면 형태가 왜곡됩니다. 현재 엔진은 비율을 유지해 축소/확대한 뒤 빈 부분을 검정으로 채우는 **resize+pad**를 씁니다. 이에 따라 실제 제품이 차지하는 픽셀 수와 작은 자국의 가시성이 바뀔 수 있습니다.
원본 크기(raw)와 EXIF 방향 보정 후 크기는 구분합니다. 크기8종이 제품8종이라는 뜻은 아닙니다.

In [ ]:
resolution = pd.crosstab([train.width, train.height], train.label.astype(int)).reindex(columns=[0, 1], fill_value=0)
resolution.columns = ["정상", "불량"]
display(resolution.rename_axis(["원본 가로", "원본 세로"]))
print("학습 해상도 종류:", len(resolution))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
resolution.plot.bar(stacked=True, color=["#268b8d", "#d79635"], ax=axes[0], legend=False)
axes[0].set(title="Raw sizes by label", xlabel="Width, height", ylabel="Images")
axes[1].hist([train.aspect_raw, test.aspect_raw], bins=15, label=["Train", "Test"], alpha=.7)
axes[1].set(title="Input aspect ratio (metadata only)", xlabel="Raw width / height", ylabel="Images")
axes[1].legend()
plt.tight_layout()
plt.show()
display(pd.DataFrame({"role": ["train", "test"], "raw size kinds": [train.size_raw.nunique(), test.size_raw.nunique()]}))
if {"rgb_width", "rgb_height"}.issubset(df.columns):
    oriented_change = (df.width.ne(df.rgb_width) | df.height.ne(df.rgb_height)) & df.rgb_width.notna() & df.rgb_height.notna()
    print("EXIF 보정으로 크기가 바뀐 기록:", int(oriented_change.sum()))

## 5. 품질 지표는 후보를 찾는 도구

아래 값은 카탈로그의 이미지별 RGB 평균입니다. 물리적 노출량·블러·실제 결함을 직접 측정한 값이 아닙니다.
이 차트만 보고 이미지를 삭제하거나 라벨을 바꾸지 않습니다. 디코드 실패 0과 품질 문제 없음도 다릅니다.

### 이 절은 ‘파일이 읽히는가’와 ‘사진이 판단에 유용한가’를 나눕니다
**decode(디코드)**는 JPEG/PNG 같은 저장 내용을 픽셀로 풀어 읽는 과정입니다. 디코드 실패0은 당시 검사에서 픽셀 읽기 오류가 없었다는 뜻입니다. 흐림·과노출·가림 때문에 결함을 볼 수 없는 사진도 디코드는 성공할 수 있습니다. 파일 확장자가.jpg인지만 확인한 검사보다 범위가 넓지만, 의미적 품질을 보장하지는 않습니다.

| 중복의 수준 | 무엇이 같은가? | 현재 상태 |
|---|---|---|
| 바이트 중복 | 파일 내용이 완전히 같음 | 과거 조사0그룹 |
| 픽셀 중복 | 방향·색상 규약으로 풀었을 때 픽셀이 같음; 파일 메타데이터는 다를 수 있음 | `unassessed`: 아직 이 검사를 하지 않음 |
| 근접 중복 | 재압축·작은 이동·노출 변화 등으로 아주 비슷함 | `unassessed`: 아직 관계 후보 조사를 하지 않음 |

`unassessed`는0개·정상·실패가 아니라 **미조사**입니다. 현재 profile 코드는 바이트 중복만 계산하고 나머지는 이 상태로 명시합니다. 관계 후보를 찾고 확인하는 후속 작업이 남아 있습니다.
RGB는 빨강·초록·파랑 채널입니다. 평균은 화면의 밝기/색 구성에 관한 기술적 단서이고 표준편차는 값의 퍼짐입니다. 결함 수나 실제 조명 강도·블러의 정답은 아닙니다.

In [ ]:
display(pd.DataFrame([
    {"항목": "과거 디코드 실패", "값": len(profile.get("decode_failures", []))},
    {"항목": "과거 바이트 중복 그룹", "값": len(profile.get("exact_duplicate_groups", []))},
    {"항목": "픽셀 중복 조사", "값": profile.get("pixel_duplicate_status", "unknown")},
    {"항목": "근접 중복 조사", "값": profile.get("near_duplicate_status", "unknown")},
]))
display(df.groupby("role")[["width", "height", "mode"]].agg(lambda s: int(s.isna().sum())).rename_axis("누락 수"))
# decode_error의 null은 오류가 기록되지 않았다는 뜻이며 누락 문제와 구분합니다.
fig, ax = plt.subplots(figsize=(7, 3))
for label, color in [(0, "#268b8d"), (1, "#d79635")]:
    ax.hist(train.loc[train.label.eq(label), "mean_rgb"].dropna(), bins=25, alpha=.6, label=f"Label {label}", color=color)
ax.set(title="Per-image RGB mean (descriptive proxy)", xlabel="Mean of RGB channels, 0–1", ylabel="Images")
ax.legend()
plt.show()

## 6. 실제로 어디까지 관측했나요?

카탈로그 확인과 별도로 **원본을 본 기록**을 읽습니다. 준비 시점의 `visual_readings=0`은 관측 전 기록입니다.
후속 viewing receipt의 수와 첫·둘째 원응답 해시가 맞는지 확인합니다. 근거가 없거나 바뀌었으면 수를 확정하지 않습니다.
두 번째 판독의 16장은 첫 48장의 일부이므로 **64개 서로 다른 이미지를 관측했다고 더하지 않습니다.**

### 이 절은 ‘관측 범위’를 다룹니다
원본 검사, 시각 판독, 독립 재판독, 전문가 정답 검증은 다른 작업입니다.
- 첫48장은 “실제로 이 사진들을 열어 기술했다”는 기록입니다.
- 두 번째16장은 그중 일부의 재판독입니다. 서로 다른 총64장이 아닙니다.
- 원래 라벨과 대조하지 않은 기술 관측이므로, “AI가 정상/불량을 몇 개 맞혔다”는 결과는 없습니다.
- 전문가 정답 검증 false는 현장 전문가가 실제 결함임을 확인한 결과가 없다는 뜻입니다.

In [ ]:
first = read_evidence(OBS_DIR / "first-reader-observations-002.json")
second = read_evidence(OBS_DIR / "second-reader-raw.json")
viewing = read_evidence(OBS_DIR / "viewing-receipt-002.json")
comparison = read_evidence(OBS_DIR / "reader-comparison-001.json")
observed = pd.DataFrame()
other = pd.DataFrame()
if first is not None and second is not None and viewing is not None and comparison is not None:
    first_hash = sha256_file(OBS_DIR / "first-reader-observations-002.json")
    second_hash = sha256_file(OBS_DIR / "second-reader-raw.json")
    links = [(viewing, "first_observations_sha256", first_hash),
             (viewing, "second_observations_sha256", second_hash),
             (comparison, "first_reader_sha256", first_hash),
             (comparison, "second_reader_sha256", second_hash)]
    for source, field, value in links:
        if source.get(field) != value:
            raise ValueError("판독 근거 해시 누락/불일치. 현재 유효한 판독 수로 해석하지 마세요.")
    observed = pd.DataFrame(first["rows"])
    other = pd.DataFrame(second["observations"])
    if observed.survey_id.duplicated().any() or other.survey_id.duplicated().any():
        raise ValueError("익명 판독 ID가 중복됩니다.")
    if not set(other.survey_id).issubset(set(observed.survey_id)):
        raise ValueError("재판독 명부가 첫 판독 집합에 포함되지 않습니다.")
    if len(observed) != viewing["first_reader_original_copy_count"] or len(other) != viewing["independent_second_reader_original_copy_count"]:
        raise ValueError("원응답 수와 viewing receipt 수가 다릅니다.")
    display(pd.DataFrame([
        {"관측": "첫 판독 원본 사본", "이미지 수": len(observed), "의미": "시각적 기술 관측"},
        {"관측": "지정 재판독", "이미지 수": len(other), "의미": "첫 집합의 일부, 새 고유 표본이 아님"},
        {"관측": "전문가 결함 정답 검증", "이미지 수": None, "의미": str(viewing.get("field_expert_ground_truth", "unknown"))},
    ]))
    print("원래 라벨 대조:", first.get("labels_unblinded", "unknown"), "| 불량 예측 생성:", first.get("class_predictions_generated", "unknown"))
else:
    print("판독 근거가 없는 환경입니다. 해당 관측 상태는 unknown이며 아래 판독 셀은 건너뜁니다.")

## 7. 48장의 관측 내용 — 탐색적 원응답

이 표는 **첫 판독자의 원응답 수**입니다. 불균등 층화 표본이므로 전체 모집단 비율이 아니고, 자국 후보는 정상/불량 라벨이 아닙니다.
`adequate`처럼 큰 구성이 읽힌다는 응답과 미세 정보가 충분하다는 응답도 별개입니다. 규약은 아직 조정 중입니다.

### 이 절은 ‘관측 항목’을 요약합니다

| 원응답/항목 | 쉬운 뜻 | 혼동하면 안 되는 것 |
|---|---|---|
| product_count: `one` | 판독자가 사진에서 제품 하나라고 관찰 | 정상0을 뜻하지 않음 |
| product_count: `multiple` | 사진에서 제품 여러 개라고 관찰 | 불량1을 뜻하지 않음. 부품·제품을 세는 기준도 조정 필요 |
| visible_marks: `seen` | 표면에 자국 같은 후보가 보임 | 실제 결함 확인이나 라벨1이 아님 |
| `not_seen` | 해당 조건에서 자국 후보를 못 봄 | 정상0 보장 아님 |
| `unclear` | 있다고도 없다고도 확정하기 어려움 | 오류·누락·정상을 자동 대체하지 않음 |
| reflection | 빛 반사로 보이는 하이라이트 등의 관측 | 무조건 나쁜 화질·실제 흠집이라는 뜻 아님 |
| occlusion | 다른 물체 등으로 일부가 가려지는 현상 | 불량과 별개 |
| readability | 큰 구성·표기 등을 읽을 수 있는지에 대한 원응답 | 미세 결함도 충분히 보인다는 보장 아님 |
| small_detail_visibility | 작은 표면·세부 정보를 얼마나 볼 수 있는지 | 실제 결함 존재와 별개 |

이것들은 현재 **조사용 관측 속성**입니다. 머신러닝의 feature처럼 나눠 적은 항목이라고는 할 수 있지만, ResNet에 직접 입력한 수치 특징이나 학습 라벨은 아닙니다. 지금 모델 입력은 이미지이고 목표 라벨은0/1입니다.
빛 반사가 흠집처럼 보이거나 작은 자국을 가릴 수 있고, 가림·축소·질감도 해석을 바꿀 수 있어 이런 항목을 기록합니다. 다만 중요성은 실제 이미지·평가에서 확인해야 합니다.

In [ ]:
if not observed.empty:
    labels = {"product_count": "제품 수 관측", "visible_marks": "자국 후보", "readability": "가독성 원응답",
              "reflection": "반사", "occlusion": "가림", "small_detail_visibility": "미세 정보"}
    raw_counts = []
    for field, meaning in labels.items():
        for value, n in observed[field].fillna("missing").value_counts().items():
            raw_counts.append({"항목": meaning, "원응답": value, "관측 수": n, "분모": len(observed)})
    display(pd.DataFrame(raw_counts))
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    for ax, field, title in zip(axes, ["product_count", "visible_marks"], ["Observed product count", "Visible mark candidates"]):
        observed[field].value_counts().plot.bar(ax=ax, color="#268b8d", rot=0)
        ax.set(title=title + " — sample only", ylabel="Reader responses")
    plt.tight_layout()
    plt.show()

## 8. 익명 실제 이미지와 당시 관측 메모

`SHOW_IMAGES=True`로 실행하면 이미 관찰한 학습 이미지의 사본만 표시합니다. 표시 전 사본 SHA-256을 원응답과 대조합니다.
이 화면에는 **원래 라벨을 붙이지 않습니다.** 사진을 다시 표시하는 것만으로 판독 조정·결함 검증·전처리 실험이 완료되지는 않습니다.
노트북의 축소 표시는 탐색용입니다. 원본 세부 정보는 필요하면 익명 원본 사본을 별도 뷰어에서 원본 크기로 확인하세요.

### 이 절은 ‘사진과 해석의 연결’을 보여 줍니다
**익명**은 원래 파일명·정상/불량 라벨·층 정보를 숨기고 조사용 ID만 붙였다는 뜻입니다. 사진을 새로 생성하거나 내용을 지웠다는 뜻이 아닙니다. 연결표는 별도로 보관하고 이 노트북에서는 읽지 않습니다.
**당시 관측 메모**는 첫 판독자가 그때 보았다고 적은 설명입니다. 확정된 결함 설명이나 전문가 정답이 아니며, 지금 사진을 다시 띄워도 자동으로 갱신되지 않습니다.
사진과 메모를 나란히 보는 목적은 “어떤 부분을 보고 이런 말을 했는가, 반사·무늬와 혼동하지 않았는가”를 대조하기 위해서입니다. 메모가 실제 사진보다 더 강한 증거가 되지는 않습니다.

In [ ]:
if SHOW_IMAGES and not observed.empty:
    chosen = PHOTO_IDS or observed.survey_id.head(MAX_PHOTOS).tolist()
    if len(chosen) > MAX_PHOTOS or not set(chosen).issubset(set(observed.survey_id)):
        raise ValueError("기존 첫 판독 집합 안에서 MAX_PHOTOS 이하로 선택하세요.")
    image_root = (OBS_DIR / "originals").resolve()
    for sid in chosen:
        row = observed.loc[observed.survey_id.eq(sid)].iloc[0]
        matches = [p for p in image_root.iterdir() if p.stem == sid and p.is_file()]
        if len(matches) != 1:
            raise ValueError("익명 사본이 없거나 여러 개입니다.")
        path = matches[0].resolve()
        if not path.is_relative_to(image_root) or sha256_file(path) != row.source_copy_sha256:
            raise ValueError("익명 원본 사본 경로/해시 불일치. 표시 중단.")
        display(Markdown(f"### 익명 ID {sid}"))
        display(DisplayImage(data=path.read_bytes(), width=420))
        display(pd.DataFrame([{k: row[k] for k in ["product_count", "visible_marks", "mark_description", "small_detail_visibility", "evidence_notes"]}]))
    print("이번에 다시 표시한 기존 이미지 수:", len(chosen), "| 새 표본 판독/정답 검증: 수행하지 않음")
else:
    print("이미지 표시 꺼짐. 위 설정의 SHOW_IMAGES를 켜면 익명 원본 사본과 관측 메모를 볼 수 있습니다.")

## 9. 두 판독자의 비교와 미해결 사례

지정 16장의 **항목별 교차표**와 선택된 자국 차이 사례를 구분합니다. `9/16`을 결함 오류율로 계산하지 않습니다.
같은 제품 수 응답도 결함 인식 정확도가 아닙니다. `adequate`와 `good` 등 서로 다른 enum은 몰래 합치지 않습니다.
원래 라벨·첫 답변을 가린 재판독의 이력과 최초 응답은 보존합니다.

### 이 절은 ‘측정의 일관성’을 다룹니다
행은 판독자A의 응답, 열은B의 응답이고 각 칸은 그 조합의 수인 **교차표**입니다. 대각선 밖의 값은 응답 차이입니다. 다르다고 어느 쪽이 틀렸는지 바로 알 수는 없습니다.
제품 수가 같다는 것은 제품 수 항목의 일치일 뿐 결함 판별 정확도가 아닙니다. 가독성에서 `adequate`와`good`처럼 정의가 다른 표현은 같은 뜻인지 확인하기 전 합치지 않습니다.
9건은 자국 후보에 관한 선택된 차이의 근거 대조 대상입니다. 전체 오류율·정상/불량 정확도·전문가 검증으로 해석하지 않습니다.

In [ ]:
if not observed.empty and not other.empty:
    paired = observed.merge(other, on="survey_id", suffixes=("_A", "_B"), validate="one_to_one")
    print("비교 가능한 지정 재판독:", len(paired))
    for field in ["product_count", "visible_marks", "readability"]:
        display(Markdown(f"**{field} — 두 원응답의 교차표**"))
        display(pd.crosstab(paired[field + "_A"], paired[field + "_B"], dropna=False))
    differences = paired.loc[paired.visible_marks_A.ne(paired.visible_marks_B), ["survey_id", "visible_marks_A", "visible_marks_B"]]
    display(Markdown("**자국 후보 응답 차이 — 조정 전**"))
    display(differences.reset_index(drop=True))
    print("현재 조정 상태:", comparison.get("adjudication_status", "unknown"))
    print("결함 정확도:", comparison.get("defect_accuracy"), "| 전체 모집단 신뢰도:", comparison.get("population_reliability"))
    print("가독성 규약 차이:", comparison.get("readability_schema_difference", "unknown"))

## 10. 아직 하지 않은 일과 다음 조사

| 구분 | 현재 상태 | 다음에 바꿀 결정 |
|---|---|---|
| 9건 근거 대조·가독성 정의 조정 | 미완료 | 표기 매핑인지 의미 변화인지 판단 |
| 실제 engine의 256/384/512 비교 | 미실행 | 입력에서 정보가 남는지 확인 |
| 추가 192장·발견 29장 판독 | 미실행, 조건부 | 더 넓은 빈도/새 실패 유형이 필요한가 |
| 픽셀·근접중복·촬영 관계 조사 | 미완료 | 그룹과 검증 경계 |
| 최종 분할과 일반화 목표 | 미확정 | 새 촬영/제품/결함 중 평가 대상 |
| 실제 대회 데이터 학습·성능 | 미실행 | 모델 비교와 비용 |

표본 수는 작업 예산입니다. 큰 표본·두 AI의 합의·깨끗한 파일만으로 일반화를 보장하지 않습니다.
노트북의 통계 셀을 실행했다고 위 단계들이 완료되는 것은 아닙니다.

### 이 절은 ‘현재 위치와 다음 결정’을 다룹니다
우리는 기술적 전수 목록과 작은 표본 관측을 확보했고, **관측 정의를 맞추는 단계**에 있습니다. 다음은 같은 원본의 실제 입력 변환 비교입니다. 이어 필요한 관계 조사와 목표 정의로 검증 분할을 정합니다.
여기서 새 촬영에 일반화한다는 것은 “이미 본 사진과 비슷한 것을 잘 맞힌다”와 다릅니다. 실제로 다른 촬영/개체가 검증에 남아 있어야 하고, 새 제품·새 결함에 대한 주장에는 그에 맞는 자료와 분할이 필요합니다.

## 11. 연구·도구 구축까지 포함한 작업 이력

- 입력 계약, 원본 검증·카탈로그, Python 학습/평가/재개 경로를 구축했습니다. 기능 검사는 작은 합성 CPU 자료에서 수행한 범위입니다.
- 전수 파일·픽셀 조사 기록과 학습/테스트 수를 확보했고, 첫 판독48/지정 재판독16의 기술 관측을 보존했습니다.
- 표본48⊂240, 발견 표본, 원래 라벨을 가린 판독과 재판독 규약을 설계했습니다.
- 산업 불량 판별의 기초·2025/2026 문헌을 조사하고 GPT-6 Pro로 연구 질문·EDA 방법을 두 차례 검토했습니다. 문헌 자문은 실제 데이터 분석과 별개입니다.
- Pro에는 연구용 텍스트·코드를 제공했습니다. 두 문헌조사 요청에 실제 원본 이미지는 첨부하지 않았습니다.
- 연구용 이미지 제공·LLM 증강은 사용자 설명상 허용됩니다. 실제 생성·추가 학습·대회 제출은 수행하지 않았습니다.

참고: [조사 계획](../docs/research/IMAGE-DATA-AUDIT-PLAN.md), [첫 표본 규약](../docs/research/EDA-FIRST-LOOK.md), [연구 논문 안내](../docs/research/INDUSTRIAL-DEFECT-PAPER-GUIDE.md).
비공개 Pro 응답은 로컬 `.insane-review/`에 있으며 공개 저장소에는 포함하지 않습니다.

### 이 절은 ‘연구 이력’을 다룹니다
도구가 구현됐다는 사실은 데이터 분석 완료나 성능 확보와 다릅니다. 문헌 검토는 선택할 질문의 근거이고, 기능 테스트는 프로그램 경로가 작동하는지 보는 검사입니다. 실제 대회 AUROC는 이후 정해진 검증 예측에서 계산해야 합니다.

## 12. 이번 노트북이 읽은 근거

해시는 **이번 실행 시 읽은 파일**을 식별합니다. 원본 재검증·전문가 검증·원격 저장·복구를 대신하지 않습니다.
실행본을 공유하기 전 이미지·개별 관측·경로와 출력이 포함되었는지 확인하세요. 공개 원본은 출력 없이 유지합니다.

### 이 절은 ‘재현과 추적’을 다룹니다
어떤 카탈로그와 어느 판독 버전을 읽었는지 남겨야 나중에 같은 표를 다시 만들 수 있습니다. 해시는 재료를 식별하는 값입니다. 실험 결과의 타당성, 관측자의 정확도, 원본 라벨의 진실성을 자동으로 보장하지 않습니다.

In [ ]:
display(pd.DataFrame([{"읽은 근거": k, "SHA-256": v} for k, v in INPUT_HASHES.items()]))
print("이 노트북이 원본/판독 파일에 쓴 내용: 없음")